# Karatsuba Length Generalization — Phase 5
**Approach:** Take the 98% 8-bit setup (hierarchical concat + DFS + grokking) and make ONE fix:
replace learned embeddings with sinusoidal for depth, sub_problem_id, and timestep.
Also add input injection (cheap, proven helpful).

**Runtime:** A100 GPU. Runtime → Change runtime type → A100.

In [ ]:
# Cell 1: Setup
import os
if not os.path.exists('/content/recursive-transformers/src'):
    !git clone https://github.com/dhruvsyam123/recursive-transformers.git /content/recursive-transformers
else:
    !cd /content/recursive-transformers && git pull
%cd /content/recursive-transformers
!pip install -q jax[cuda12] equinox optax jaxtyping numpy pyyaml matplotlib einops

import jax
print(f'JAX devices: {jax.devices()}')
print(f'JAX backend: {jax.default_backend()}')

In [ ]:
# Cell 2: Model — All-sinusoidal hierarchical positions + input injection
# Changes from Phase 3c (98% run):
# 1. depth_encoder: learned Embedding → sinusoidal (generalizes to depth 2+)
# 2. sub_problem_encoder: learned Embedding → sinusoidal (generalizes to IDs 4+)
# 3. step_type_encoder: learned Embedding → sinusoidal (consistency)
# 4. timestep: learned Embedding → sinusoidal (generalizes to loops 8+)
# 5. Input injection: add x0 back at each loop (prevents forgetting)
# Everything else identical to Phase 3c.

import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import math

from src.model.looped_transformer import (
    MultiHeadSelfAttention, FeedForward, RMSNorm, count_parameters,
)


def sinusoidal_encoding(positions, d_model, max_len=512):
    """Fixed sinusoidal encoding — works for ANY position value."""
    pos = positions.astype(jnp.float32)
    if pos.ndim == 0:
        pos = pos[None]
        squeeze = True
    else:
        squeeze = False
    dim_indices = jnp.arange(0, d_model, 2, dtype=jnp.float32)
    freqs = jnp.exp(-dim_indices * (math.log(10000.0) / d_model))
    angles = pos[:, None] * freqs[None, :]
    enc = jnp.zeros((pos.shape[0], d_model))
    enc = enc.at[:, 0::2].set(jnp.sin(angles))
    enc = enc.at[:, 1::2].set(jnp.cos(angles))
    if squeeze:
        return enc[0]
    return enc


class AllSinusoidalHierarchicalPos(eqx.Module):
    """Hierarchical position encoding with ALL components sinusoidal (not learned).
    Concat mode: each component gets d_model//4 dimensions.
    Generalizes to unseen values of depth, sub_problem_id, etc."""
    d_model: int = eqx.field(static=True)
    comp_dim: int = eqx.field(static=True)

    def __init__(self, d_model):
        assert d_model % 4 == 0
        self.d_model = d_model
        self.comp_dim = d_model // 4

    def __call__(self, bit_sig, depth, sub_id, step_type):
        return jnp.concatenate([
            sinusoidal_encoding(bit_sig, self.comp_dim),
            sinusoidal_encoding(depth, self.comp_dim),
            sinusoidal_encoding(sub_id, self.comp_dim),
            sinusoidal_encoding(step_type, self.comp_dim),
        ], axis=-1)


class LoopBlock(eqx.Module):
    """Transformer block with SINUSOIDAL timestep (not learned)."""
    attention: MultiHeadSelfAttention
    ffn: FeedForward
    norm1: RMSNorm
    norm2: RMSNorm
    d_model: int = eqx.field(static=True)

    def __init__(self, d_model, n_heads, d_ff, *, key):
        k1, k2 = jax.random.split(key)
        self.attention = MultiHeadSelfAttention(d_model, n_heads, key=k1)
        self.ffn = FeedForward(d_model, d_ff, key=k2)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        self.d_model = d_model

    def __call__(self, x, timestep, mask=None):
        t_emb = sinusoidal_encoding(timestep, self.d_model)
        x_cond = x + t_emb[None, :]
        normed = jax.vmap(self.norm1)(x_cond)
        x = x + self.attention(normed, mask=mask)
        normed = jax.vmap(self.norm2)(x)
        x = x + jax.vmap(self.ffn)(normed)
        return x


class KaratsubaModel(eqx.Module):
    """Looped transformer with all-sinusoidal hierarchical positions + input injection.
    Identical to Phase 3c (98% run) EXCEPT:
    - All position components are sinusoidal (not learned)
    - Timestep is sinusoidal (not learned)
    - Input injection at each loop iteration
    """
    token_embed: eqx.nn.Embedding
    hier_pos: AllSinusoidalHierarchicalPos
    block: LoopBlock
    final_norm: RMSNorm
    output_head: eqx.nn.Linear
    inject_scale: jnp.ndarray

    def __call__(self, tokens, n_loops, bit_sig, depth, sub_id, step_type):
        # 1. Token embedding + hierarchical position (ALL sinusoidal, concat)
        x0 = jax.vmap(self.token_embed)(tokens)
        pos = self.hier_pos(bit_sig, depth, sub_id, step_type)
        x0 = x0 + pos  # x0 includes position info for input injection
        x = x0

        # 2. Causal mask
        seq_len = tokens.shape[0]
        mask = jnp.tril(jnp.ones((seq_len, seq_len), dtype=jnp.bool_))

        # 3. Looped transformer with input injection + sinusoidal timestep
        def scan_fn(hidden, timestep):
            hidden = hidden + self.inject_scale * x0  # input injection
            hidden = self.block(hidden, timestep, mask=mask)
            return hidden, None

        x, _ = jax.lax.scan(scan_fn, x, jnp.arange(n_loops))

        # 4. Output
        x = jax.vmap(self.final_norm)(x)
        return jax.vmap(self.output_head)(x)


D_MODEL = 256
VOCAB = 143
key = jax.random.PRNGKey(42)
k1, k2, k3 = jax.random.split(key, 3)

model = KaratsubaModel(
    token_embed=eqx.nn.Embedding(VOCAB, D_MODEL, key=k1),
    hier_pos=AllSinusoidalHierarchicalPos(D_MODEL),
    block=LoopBlock(D_MODEL, n_heads=8, d_ff=512, key=k2),
    final_norm=RMSNorm(D_MODEL),
    output_head=eqx.nn.Linear(D_MODEL, VOCAB, key=k3),
    inject_scale=jnp.array(0.1),
)
print(f'Parameters: {count_parameters(model):,}')

In [ ]:
# Cell 3: Datasets (DFS ordering) + Loss Mask + Position Processing
from src.data.dataset import MultiplicationDataset, DataConfig
from src.data.tokenizer import Tokenizer

tok = Tokenizer()
INPUT_ID = tok.encode_token('[INPUT]')
BASE_MUL_ID = tok.encode_token('[BASE_MUL]')
SPLIT_ID = tok.encode_token('[SPLIT]')

# Datasets — DFS ordering (proven to work)
ds4 = MultiplicationDataset(DataConfig(
    bit_widths=[4], algorithm='karatsuba', base_case_bits=4,
    train_fraction=1.0, seed=42,
))
ds8 = MultiplicationDataset(DataConfig(
    bit_widths=[8], algorithm='karatsuba', base_case_bits=4,
    ordering='depth_first',
    max_samples_per_width=4000, seed=42,
))
ml4 = ds4.get_max_seq_len('train') + 2
ml8 = ds8.get_max_seq_len('train') + 2
print(f'4-bit: {ds4.train_size()} examples, max_len={ml4}')
print(f'8-bit: {ds8.train_size()} train / {ds8.test_size()} test, max_len={ml8}')

# Loss mask (same as Phase 3c — proven correct)
def make_mask(token_ids_full, pad_mask_full):
    BIT_0, BIT_1 = tok.encode_token(0), tok.encode_token(1)
    is_bit = (token_ids_full == BIT_0) | (token_ids_full == BIT_1)
    seen = jnp.cumsum(
        (token_ids_full == SPLIT_ID) | (token_ids_full == BASE_MUL_ID), axis=1
    ) > 0
    predictable = ~(~seen & is_bit)
    return (pad_mask_full * predictable.astype(jnp.float32))[:, 1:]

# Position processing: fix tag token bit_significance (same as Phase 3c)
def process_positions(pos):
    bit_sig = pos[:, :, 0]
    step_type = pos[:, :, 3]
    is_tag = bit_sig < 0
    bit_sig_fixed = jnp.where(is_tag, 200 + step_type, bit_sig)
    return bit_sig_fixed, pos[:, :, 1], pos[:, :, 2], pos[:, :, 3]

In [ ]:
# Cell 4: Training — identical to Phase 3c except model internals
import random

# Optimizer (same as Phase 3c)
schedule = optax.warmup_cosine_decay_schedule(
    init_value=1e-5, peak_value=1e-3, warmup_steps=500,
    decay_steps=50000, end_value=1e-6
)
optimizer = optax.adamw(learning_rate=schedule, weight_decay=0.15)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

# Training step (same structure as Phase 3c)
def make_train_step(n_loops):
    @eqx.filter_jit
    def train_step(model, opt_state, token_ids_full, pad_mask_full, positions_full):
        input_ids = token_ids_full[:, :-1]
        target_ids = token_ids_full[:, 1:]
        inp_pos = positions_full[:, :-1]
        mask = make_mask(token_ids_full, pad_mask_full)
        def loss_fn(model):
            bit_sig, depth, sub_id, step_t = process_positions(inp_pos)
            def forward_one(ids, bs, d, si, st):
                return model(ids, n_loops, bs, d, si, st)
            logits = jax.vmap(forward_one)(input_ids, bit_sig, depth, sub_id, step_t)
            log_probs = jax.nn.log_softmax(logits, axis=-1)
            targets_oh = jax.nn.one_hot(target_ids, logits.shape[-1])
            tok_loss = -jnp.sum(targets_oh * log_probs, axis=-1)
            return jnp.sum(tok_loss * mask) / jnp.maximum(jnp.sum(mask), 1.0)
        loss, grads = eqx.filter_value_and_grad(loss_fn)(model)
        updates, new_opt = optimizer.update(grads, opt_state, eqx.filter(model, eqx.is_array))
        return eqx.apply_updates(model, updates), new_opt, loss
    return train_step

# Pre-compile for loop counts (include 10, 12 for 16-bit generalization)
train_steps = {n: make_train_step(n) for n in [4, 6, 8, 10, 12]}

# Eval function
def quick_eval(model, n_bits, n_loops, n_samples=200):
    ev = MultiplicationDataset(DataConfig(
        bit_widths=[n_bits], algorithm='karatsuba', base_case_bits=4,
        max_samples_per_width=n_samples, train_fraction=0.0, seed=999,
    ))
    ml = ev.get_max_seq_len('test') + 2
    correct, total = 0, 0
    for batch in ev.get_batch(split='test', batch_size=16, shuffle=False, max_len=ml):
        tk = jnp.array(batch['token_ids'])
        pm = jnp.array(batch['mask'])
        ps = jnp.array(batch['positions'])
        inp = tk[:, :-1]
        tgt = tk[:, 1:]
        inp_pos = ps[:, :-1]
        em = make_mask(tk, pm)
        bit_sig, depth, sub_id, step_t = process_positions(inp_pos)
        def fwd(ids, bs, d, si, st):
            return model(ids, n_loops, bs, d, si, st)
        logits = jax.vmap(fwd)(inp, bit_sig, depth, sub_id, step_t)
        preds = jnp.argmax(logits, axis=-1)
        matches = (preds == tgt) | (em == 0)
        correct += int(jnp.all(matches, axis=-1).sum())
        total += len(batch['x'])
    return correct, total

# Training loop (same as Phase 3c)
rng = random.Random(42)

print('Training (all-sinusoidal positions + input injection + DFS)...')
for epoch in range(500):
    epoch_loss, n_batches = 0.0, 0

    # 4-bit batches
    n_loops = rng.choice([4, 6, 8])
    step_fn = train_steps[n_loops]
    for batch in ds4.get_batch(split='train', batch_size=64, max_len=ml4):
        tk = jnp.array(batch['token_ids'])
        pm = jnp.array(batch['mask'])
        ps = jnp.array(batch['positions'])
        model, opt_state, loss = step_fn(model, opt_state, tk, pm, ps)
        epoch_loss += float(loss)
        n_batches += 1

    # 8-bit batches (include higher loop counts for timestep coverage)
    n_loops = rng.choice([4, 6, 8, 10, 12])
    step_fn = train_steps[n_loops]
    for batch in ds8.get_batch(split='train', batch_size=16, max_len=ml8):
        tk = jnp.array(batch['token_ids'])
        pm = jnp.array(batch['mask'])
        ps = jnp.array(batch['positions'])
        model, opt_state, loss = step_fn(model, opt_state, tk, pm, ps)
        epoch_loss += float(loss)
        n_batches += 1

    if epoch % 50 == 0:
        avg = epoch_loss / n_batches
        c8, t8 = quick_eval(model, 8, 8)
        print(f'Epoch {epoch:4d} | Loss: {avg:.4f} | 8-bit: {c8}/{t8} = {c8/t8:.1%}')

print('\nTraining complete.')
c8, t8 = quick_eval(model, 8, 8)
print(f'Final 8-bit: {c8}/{t8} = {c8/t8:.1%}')

In [ ]:
# Cell 5: Length generalization evaluation
print('=' * 50)
print('LENGTH GENERALIZATION EVALUATION')
print('All-Sinusoidal Positions + Input Injection + DFS')
print('=' * 50)
print(f"\n{'Bits':>6} {'Loops':>6} {'Correct':>8} {'Total':>6} {'Accuracy':>10}")
print('-' * 44)

for n_bits, n_loops in [(4, 8), (8, 8), (16, 12), (32, 16)]:
    try:
        n_samples = 200 if n_bits <= 16 else 50
        c, t = quick_eval(model, n_bits, n_loops, n_samples=n_samples)
        print(f'{n_bits:6d} {n_loops:6d} {c:8d} {t:6d} {c/t:10.1%}')
    except Exception as e:
        print(f'{n_bits:6d} {n_loops:6d} {"ERROR":>8} {"":>6} {str(e)[:50]}')

print('\n' + '=' * 50)

---
## Option A: Recursive Evaluator with Neural Subroutine
The model achieves 99% on 8-bit but 0% on 16-bit as a flat sequence.
Here we use the model as a **learned multiplication subroutine** within
classical Karatsuba recursion. The model handles each bounded-size step;
the recursion is managed externally.

This eliminates the autoregressive cascade (sequences stay short),
attention OOD (same lengths as training), and compositional gap
(each call is identical to training).

In [ ]:
# Cell 7: Recursive evaluator — neural Karatsuba
from src.data.karatsuba_trace import KaratsubaTraceGenerator, int_to_bits, bits_to_int

gen = KaratsubaTraceGenerator(base_case_bits=4)

def model_multiply(model, x, y, n_bits, n_loops=8):
    """Use the trained model to multiply x * y (n_bits <= 8).
    Generates the full trace, runs teacher-forced prediction,
    extracts the product from model output."""
    assert n_bits <= 8, f'model_multiply only handles <= 8-bit, got {n_bits}'
    assert x < (1 << n_bits) and y < (1 << n_bits)

    # Generate trace and tokenize
    trace = gen.generate(x, y, n_bits, ordering='depth_first')
    seq = gen.trace_to_token_sequence(trace)
    token_ids, positions = tok.encode_trace_sequence(seq)
    seq_len = len(token_ids)

    # Prepare padded inputs
    max_len = seq_len + 1
    padded_ids, padded_pos, mask = tok.pad_sequence(token_ids, positions, max_len)
    tk = jnp.array(padded_ids)[None, :]  # (1, max_len)
    ps = jnp.array(padded_pos)[None, :]  # (1, max_len, 4)

    # Process positions (same as training)
    bit_sig, depth, sub_id, step_t = process_positions(ps[:, :-1])

    # Run model (teacher-forced)
    def fwd(ids, bs, d, si, st):
        return model(ids, n_loops, bs, d, si, st)
    logits = jax.vmap(fwd)(tk[:, :-1], bit_sig, depth, sub_id, step_t)
    preds = jnp.argmax(logits, axis=-1)[0]  # (seq_len,)

    # Find the LAST [OUTPUT] tag and extract product bits after it
    OUTPUT_ID = tok.encode_token('[OUTPUT]')
    target_ids = jnp.array(padded_ids[1:seq_len])  # shifted targets
    pred_ids = preds[:seq_len - 1]

    # Find last [OUTPUT] in the target sequence
    output_positions = [i for i in range(len(token_ids) - 1) if token_ids[i] == OUTPUT_ID]
    if not output_positions:
        return None  # shouldn't happen

    last_output_pos = output_positions[-1]
    product_bits = 2 * n_bits

    # Extract predicted bits after the last [OUTPUT]
    BIT_0 = tok.encode_token(0)
    predicted_bits = []
    for i in range(product_bits):
        pos = last_output_pos + i  # position in the target sequence (shifted by 1)
        if pos < len(pred_ids):
            pred_token = int(pred_ids[pos])
            if pred_token == BIT_0:
                predicted_bits.append(0)
            else:
                predicted_bits.append(1)
        else:
            predicted_bits.append(0)

    return bits_to_int(predicted_bits)


def neural_karatsuba(x, y, n_bits, model, base_bits=8, n_loops=8):
    """Karatsuba multiplication using the trained model as base-case subroutine.

    For n_bits <= base_bits: calls the model directly.
    For n_bits > base_bits: classical Karatsuba decomposition, recursive calls.
    """
    if n_bits <= base_bits:
        return model_multiply(model, x, y, n_bits, n_loops)

    # Karatsuba decomposition (classical)
    half = n_bits // 2
    hi_bits = n_bits - half
    mask_low = (1 << half) - 1

    x_lo = x & mask_low
    x_hi = x >> half
    y_lo = y & mask_low
    y_hi = y >> half

    # Round sub-problem sizes to powers of 2 (model expects this)
    def next_pow2(n):
        p = 1
        while p < n:
            p <<= 1
        return max(p, 4)  # minimum 4-bit

    z0_bits = next_pow2(half)
    z2_bits = next_pow2(hi_bits)

    # z0 = x_lo * y_lo
    z0 = neural_karatsuba(x_lo, y_lo, z0_bits, model, base_bits, n_loops)
    if z0 is None: return None

    # z2 = x_hi * y_hi
    z2 = neural_karatsuba(x_hi, y_hi, z2_bits, model, base_bits, n_loops)
    if z2 is None: return None

    # z1_raw = (x_lo + x_hi) * (y_lo + y_hi)
    sum_x = x_lo + x_hi
    sum_y = y_lo + y_hi
    sum_actual_bits = max(half, hi_bits) + 1
    z1_bits = next_pow2(sum_actual_bits)

    z1_raw = neural_karatsuba(sum_x, sum_y, z1_bits, model, base_bits, n_loops)
    if z1_raw is None: return None

    # Classical combine
    z1 = z1_raw - z0 - z2
    result = z0 + (z1 << half) + (z2 << (2 * half))
    return result


# Quick test on 8-bit (should match model accuracy)
print('Testing neural_karatsuba on 8-bit...')
import random
rng = random.Random(123)
correct, total = 0, 0
for _ in range(100):
    a = rng.randint(0, 255)
    b = rng.randint(0, 255)
    result = neural_karatsuba(a, b, 8, model, base_bits=8)
    if result == a * b:
        correct += 1
    total += 1
print(f'8-bit: {correct}/{total} = {correct/total:.1%}')

In [ ]:
# Cell 8: Test recursive evaluator on 16, 32, 64-bit
import time

print('=' * 60)
print('RECURSIVE NEURAL KARATSUBA — LENGTH GENERALIZATION')
print('Model trained on 4-bit + 8-bit ONLY')
print('Recursion managed externally, model handles <= 8-bit steps')
print('=' * 60)

results = {}
for n_bits in [8, 16, 32, 64]:
    max_val = (1 << n_bits)
    n_samples = 200 if n_bits <= 16 else 50 if n_bits <= 32 else 20
    rng = random.Random(999)

    correct, total, errors = 0, 0, []
    t0 = time.time()

    for _ in range(n_samples):
        a = rng.randint(0, max_val - 1)
        b = rng.randint(0, max_val - 1)
        expected = a * b
        result = neural_karatsuba(a, b, n_bits, model, base_bits=8)

        if result == expected:
            correct += 1
        else:
            errors.append((a, b, expected, result))
        total += 1

    elapsed = time.time() - t0
    acc = correct / total
    results[n_bits] = acc

    print(f'\n{n_bits:3d}-bit: {correct}/{total} = {acc:.1%}  ({elapsed:.1f}s)')
    if errors and len(errors) <= 3:
        for a, b, exp, got in errors:
            print(f'  Error: {a} * {b} = {exp}, got {got}')

print('\n' + '=' * 60)
print('COMPARISON: Flat sequence vs Recursive evaluator')
print('=' * 60)
print(f"{'Method':<30} {'8-bit':>8} {'16-bit':>8} {'32-bit':>8} {'64-bit':>8}")
print('-' * 62)
print(f"{'Flat sequence (Cell 5)':<30} {'99.0%':>8} {'0.0%':>8} {'N/A':>8} {'N/A':>8}")
print(f"{'Recursive evaluator':<30}", end='')
for nb in [8, 16, 32, 64]:
    if nb in results:
        print(f'{results[nb]:>7.1%} ', end='')
    else:
        print(f'{"N/A":>8}', end='')
print()

In [ ]:
# Cell 6: Save model to Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
CKPT_DIR = '/content/drive/MyDrive/karatsuba_checkpoints/'
os.makedirs(CKPT_DIR, exist_ok=True)
eqx.tree_serialise_leaves(os.path.join(CKPT_DIR, 'model_phase5.eqx'), model)
print(f'Model saved to {CKPT_DIR}')